In [1]:
print("==== Mostrador del Modelo de gasto ====")

==== Mostrador del Modelo de gasto ====


In [2]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
    
print(RAIZ)
print("CSV ventas:", (RAIZ / "data" / "raw" / "ventas.csv").exists())
print("Bosque:", (RAIZ / "modelos" / "gasto_sklearn.joblib").exists())
print("Red:", (RAIZ / "modelos" / "gasto_mlp.joblib").exists())
print("Lookup:", (RAIZ / "modelos" / "features_clientes.csv").exists())
print("Meta:", (RAIZ / "modelos" / "meta.json").exists())
print("features_modelo:", (RAIZ / "src" / "features_modelo.py").exists())
print("carga_modelos:", (RAIZ / "src" / "carga_modelos.py").exists())
print("prediccion:", (RAIZ / "src" / "prediccion.py").exists())
print("entrenar:", (RAIZ / "scripts" / "entrenar_modelos.py").exists())
print("app:", (RAIZ / "app.py").exists())
print("app_hola:", (RAIZ / "app_hola.py").exists())

c:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_3\Streamlit
CSV ventas: True
Bosque: True
Red: True
Lookup: True
Meta: True
features_modelo: True
carga_modelos: True
prediccion: True
entrenar: True
app: True
app_hola: True


# EL modelo ya en disco

In [ ]:
from src.features_modelo import COLUMNA_TARGET, COLUMNAS_FEATURES

carpeta = RAIZ / "modelos"
esperados = [
    "gasto_sklearn.joblib",
    "gasto_mlp.joblib",
    "features_clientes.csv",
    "meta.json",
]

print("Carpeta:", carpeta)
for nombre in esperados:
    ruta = carpeta / nombre
    print(nombre, ruta.exists())
    print("  KB:", round(ruta.stat().st_size / 1024, 1))

Carpeta: c:\Programacion\DataScienceIA\Docente_Dev_Senior_code\Clases_Master_IA\Modulo_2\Unidad_3\Streamlit\modelos
gasto_sklearn.joblib True
  KB: 853.1
gasto_mlp.joblib True
  KB: 17.6
features_clientes.csv True
  KB: 321.2
meta.json True
  KB: 0.4


In [4]:
import pandas as pd

from src.carga_modelos import features_de_cliente
from src.features_modelo import COLUMNA_TARGET, COLUMNAS_FEATURES

ruta = RAIZ / "modelos" / "features_clientes.csv"
lookup = pd.read_csv(ruta)

print("Lookup:", ruta.exists())
print("Filas (1 por cliente):", len(lookup))
print("Tiene las 4 columnas:", all(c in lookup.columns for c in COLUMNAS_FEATURES))
print("Tiene el target:", COLUMNA_TARGET in lookup.columns)

cliente_id = 8
fila = lookup.loc[lookup["cliente_id"] == cliente_id]
print("\nUna fila del CSV (incluye el target):")
print(fila.to_string(index=False))

visto = features_de_cliente(lookup.set_index("cliente_id"), cliente_id)
print("\nLo que ve el modelo (sin", COLUMNA_TARGET + "):")
print(visto)


Lookup: True
Filas (1 por cliente): 9996
Tiene las 4 columnas: True
Tiene el target: True

Una fila del CSV (incluye el target):
 cliente_id  num_compras  gasto_historico  ticket_medio  cantidad_total  gasto_siguiente_periodo
          8           52         23601.56        453.88             112                  1956.32

Lo que ve el modelo (sin gasto_siguiente_periodo):
{'num_compras': 52, 'gasto_historico': 23601.56, 'ticket_medio': 453.88, 'cantidad_total': 112}


In [5]:
import json
import joblib

from src.carga_modelos import cargar_lookup, cargar_paquete
from src.features_modelo import COLUMNA_TARGET, COLUMNAS_FEATURES

ruta = RAIZ / "modelos" / "gasto_sklearn.joblib"
paquete = joblib.load(ruta)
sklearn = cargar_paquete("gasto_sklearn.joblib")

print("Archivo:", ruta.name)
print("Keys:", list(paquete.keys()))
for clave in paquete:
    valor = paquete[clave]
    if clave == "modelo":
        print(clave, "->", type(valor).__name__)
    else:
        print(clave, "->", valor)

lookup = cargar_lookup()
meta = json.loads((RAIZ / "modelos" / "meta.json").read_text(encoding="utf-8"))

print("\nValidacion")
print("cargar_paquete mismas keys:", list(sklearn.keys()) == list(paquete.keys()))
print("features = contrato:", paquete["features"] == COLUMNAS_FEATURES)
print("target = contrato:", paquete["target"] == COLUMNA_TARGET)
print("lookup clientes:", len(lookup), "indice:", lookup.index.name)
print("cliente 8 en lookup:", 8 in lookup.index)
print("meta version:", meta["version"], "features ok:", meta["features"] == COLUMNAS_FEATURES)


Archivo: gasto_sklearn.joblib
Keys: ['modelo', 'features', 'target', 'version', 'metricas']
modelo -> RandomForestRegressor
features -> ['num_compras', 'gasto_historico', 'ticket_medio', 'cantidad_total']
target -> gasto_siguiente_periodo
version -> 1.0.0
metricas -> {'mae': 1016.89, 'r2': 0.049, 'segundos': 0.26}

Validacion
cargar_paquete mismas keys: True
features = contrato: True
target = contrato: True
lookup clientes: 9996 indice: cliente_id
cliente 8 en lookup: True
meta version: 1.0.0 features ok: True


# Widgets y el rerun

In [ ]:
#Pinta
#Devuelve


nombre = "Jorge"
compras = 10
gasto = 2000

print(f"Hola {nombre}: {compras} compras y {gasto:,} E de historial")


Hola Jorge: 10 compras y 2,000 E de historial


In [ ]:
from src.carga_modelos import cargar_lookup, cargar_paquete, features_de_cliente
from src.prediccion import predecir

bosque = cargar_paquete("gasto_sklearn.joblib")
red = cargar_paquete("gasto_mlp.joblib")
fichas = cargar_lookup()

cifras = features_de_cliente(fichas, 8)
real = float(fichas.loc[8, "gasto_siguiente_periodo"])

v_bosque = predecir(bosque, **cifras)
v_red = predecir(red, **cifras)

print("Cliente 8, pasado:", cifras)
print(f"Bosque ---> {v_bosque:,.2f}")
print(f"Red (MLP) ---> {v_red:,.2f} ")
print(f"Real 90 d ---> {real:,.2f} ")

Cliente 8, pasado: {'num_compras': 52, 'gasto_historico': 23601.56, 'ticket_medio': 453.88, 'cantidad_total': 112}
Bosque ---> 2,017.90
Red (MLP) ---> 2,440.96 
Real 90 d ---> 1,956.32 


In [10]:
from urllib.error import URLError
from urllib.request import urlopen

url = "http://127.0.0.1:8501"

try:
    with urlopen(url, timeout=2.0) as respuesta:
        print("Streamlit está arriba. Status:", respuesta.status)

except URLError:
    print("Nadie en el 8501")

Streamlit está arriba. Status: 200
